# Projectile Motion — Animated Demo

A particle is launched with speed $v_0$ at an angle $\theta$ above the horizontal.

**Set the launch speed and angle** — the launch arrow pitches to match, and the dashed line shows the parabola the particle will trace out. Then press **Run** to watch it fly.

While the particle flies, three arrows travel with it:

- **Blue** — horizontal velocity $v_x$ (never changes)
- **Green** — vertical velocity $v_y$ (shrinks on the way up, vanishes at the top, grows downward on the way down)
- **Red** — the actual velocity $\vec{v}$ (always tangent to the path)

$$x = v_0\cos\theta\;t \qquad\qquad y = v_0\sin\theta\;t - \tfrac{1}{2} g t^2$$

$$v_x = v_0\cos\theta \qquad\qquad v_y = v_0\sin\theta - g t$$

**Things to try:**

- $\theta = 45°$ gives the greatest range.
- Complementary angles ($30°$ and $60°$) land at the **same** spot — check the readout.
- Watch the green arrow at the very top of the path: $v_y = 0$ for an instant, so the red arrow is purely horizontal there.

Air resistance is ignored and $g = 9.81\ \text{m/s}^2$.


In [ ]:
import numpy as np

import matplotlib.pyplot as plt

from matplotlib.patches import FancyArrowPatch

from matplotlib.lines import Line2D

from matplotlib.animation import FuncAnimation

try:
    import ipywidgets as widgets
except ModuleNotFoundError:
    %pip install -q ipywidgets
    import ipywidgets as widgets

from IPython.display import display, HTML





g = 9.81   # m/s^2





# ------------------------------------------------------------

# The physics

# ------------------------------------------------------------



def flight(v0, angle_deg):

    """Key quantities for a projectile launched from ground level."""

    th = np.radians(angle_deg)

    vx0 = v0 * np.cos(th)

    vy0 = v0 * np.sin(th)

    T = 2 * vy0 / g          # time of flight (lands back at y = 0)

    H = vy0**2 / (2 * g)     # max height

    R = vx0 * T              # range

    return dict(theta=th, vx0=vx0, vy0=vy0, T=T, H=H, R=R)





# ------------------------------------------------------------

# Drawing

# ------------------------------------------------------------



def build_projectile_figure(v0, angle_deg):

    """Set up the axes, the dashed parabola and all the artists."""

    fl = flight(v0, angle_deg)

    R, H = fl["R"], fl["H"]

    diag = np.hypot(R, H)

    scale = 0.15 * diag / v0          # velocity (m/s) -> arrow length (m)



    fig, ax = plt.subplots(figsize=(9.5, 4.8))

    ax.set_xlim(-0.04 * diag, R + fl["vx0"] * scale + 0.06 * diag)

    # room below ground for the downward-pointing arrows at the landing instant

    ax.set_ylim(-(fl["vy0"] * scale + 0.06 * diag),

                max(H, fl["vy0"] * scale) + 0.10 * diag)

    ax.set_aspect("equal")            # so arrows point in the true direction



    # the ground

    ax.axhline(0, color="black", linewidth=2)



    # the full parabola the particle will trace out

    ts = np.linspace(0, fl["T"], 200)

    ax.plot(fl["vx0"] * ts, fl["vy0"] * ts - 0.5 * g * ts**2,

            "k--", linewidth=1.2, alpha=0.5, zorder=1)



    # apex and landing markers

    ax.plot([R / 2], [H], marker="+", markersize=12, color="gray", zorder=1)

    ax.plot([R], [0], marker="x", markersize=10, color="gray", zorder=1)



    # artists that move / change

    trail = ax.plot([], [], color="black", linewidth=2, zorder=2)[0]

    particle = ax.plot([], [], "o", color="black", markersize=9, zorder=5)[0]



    vx_arrow = FancyArrowPatch((0, 0), (0, 0), arrowstyle="-|>", mutation_scale=16,

                               linewidth=2.2, color="tab:blue", zorder=4)

    vy_arrow = FancyArrowPatch((0, 0), (0, 0), arrowstyle="-|>", mutation_scale=16,

                               linewidth=2.2, color="tab:green", zorder=4)

    v_arrow = FancyArrowPatch((0, 0), (0, 0), arrowstyle="-|>", mutation_scale=16,

                              linewidth=2.2, color="tab:red", zorder=4)

    launch_arrow = FancyArrowPatch((0, 0), (0, 0), arrowstyle="-|>",

                                   mutation_scale=20, linewidth=2.5,

                                   color="tab:red", zorder=4)

    for a in (vx_arrow, vy_arrow, v_arrow, launch_arrow):

        ax.add_patch(a)



    launch_label = ax.text(0, 0, "", color="tab:red", fontsize=12,

                           ha="left", va="bottom", zorder=4)

    info = ax.text(0.02, 0.97, "", transform=ax.transAxes,

                   ha="left", va="top", fontsize=11,

                   bbox=dict(facecolor="white", alpha=0.85, edgecolor="lightgray"))



    legend_handles = [

        Line2D([0], [0], color="tab:blue", lw=2.2, label="$v_x$ (constant)"),

        Line2D([0], [0], color="tab:green", lw=2.2, label="$v_y$"),

        Line2D([0], [0], color="tab:red", lw=2.2, label="$\\vec{v}$"),

        Line2D([0], [0], color="black", ls="--", lw=1.2, alpha=0.5, label="path"),

    ]

    ax.legend(handles=legend_handles, loc="upper right", fontsize=10)



    ax.set_xlabel("x (m)")

    ax.set_ylabel("y (m)")

    ax.set_title(f"Launch: $v_0$ = {v0:g} m/s at $\\theta$ = {angle_deg:g}°",

                 fontsize=13)

    fig.tight_layout()



    artists = dict(

        trail=trail, particle=particle,

        vx_arrow=vx_arrow, vy_arrow=vy_arrow, v_arrow=v_arrow,

        launch_arrow=launch_arrow, launch_label=launch_label, info=info,

    )

    return fig, artists, fl, scale





def show_launch(v0, angle_deg):

    """Static preview: the pitched launch arrow and the parabola trace."""

    fig, art, fl, scale = build_projectile_figure(v0, angle_deg)



    tip = (fl["vx0"] * scale, fl["vy0"] * scale)

    art["launch_arrow"].set_positions((0, 0), tip)

    art["launch_label"].set_position((tip[0] * 1.04, tip[1] * 1.04))

    art["launch_label"].set_text(f"$v_0$ = {v0:g} m/s")

    art["vx_arrow"].set_visible(False)

    art["vy_arrow"].set_visible(False)

    art["v_arrow"].set_visible(False)



    display(fig)

    plt.close(fig)





def projectile_animation(v0, angle_deg, n_frames=48, interval=70):

    """In-browser animation: the particle flies the parabola with its vectors."""

    fig, art, fl, scale = build_projectile_figure(v0, angle_deg)

    art["launch_arrow"].set_visible(False)

    art["launch_label"].set_text("")



    ts = np.linspace(0, fl["T"], n_frames)

    xs = fl["vx0"] * ts

    ys = fl["vy0"] * ts - 0.5 * g * ts**2



    def update(i):

        t = ts[i]

        x = xs[i]

        y = max(ys[i], 0.0)

        vx = fl["vx0"]

        vy = fl["vy0"] - g * t



        art["trail"].set_data(xs[:i + 1], np.maximum(ys[:i + 1], 0.0))

        art["particle"].set_data([x], [y])

        art["vx_arrow"].set_positions((x, y), (x + vx * scale, y))

        art["vy_arrow"].set_positions((x, y), (x, y + vy * scale))

        art["vy_arrow"].set_visible(abs(vy) > 0.05)   # vanishes at the top

        art["v_arrow"].set_positions((x, y), (x + vx * scale, y + vy * scale))

        art["info"].set_text(

            f"t = {t:.2f} s\n"

            f"$v_x$ = {vx:.1f} m/s\n"

            f"$v_y$ = {vy:+.1f} m/s\n"

            f"$|\\vec{{v}}|$ = {np.hypot(vx, vy):.1f} m/s"

        )

        return tuple(art.values())



    anim = FuncAnimation(fig, update, frames=n_frames, interval=interval, repeat=False)

    plt.close(fig)

    return HTML(anim.to_jshtml())





def summary(v0, angle_deg):

    fl = flight(v0, angle_deg)

    return (f"<b>v₀ = {v0:g} m/s</b> at <b>θ = {angle_deg:g}°</b><br>"

            f"vx₀ = v₀cosθ = <b>{fl['vx0']:.1f} m/s</b> (stays constant)<br>"

            f"vy₀ = v₀sinθ = <b>{fl['vy0']:.1f} m/s</b><br>"

            f"Time of flight T = <b>{fl['T']:.2f} s</b> &nbsp;·&nbsp; "

            f"Max height H = <b>{fl['H']:.1f} m</b> &nbsp;·&nbsp; "

            f"Range R = <b>{fl['R']:.1f} m</b>")


In [ ]:
# ============================================================

# Projectile motion: set v0 and the angle, then Run

# ============================================================



speed_slider = widgets.FloatSlider(

    value=20, min=5, max=30, step=1,

    description="v₀ (m/s):", continuous_update=False

)

angle_slider = widgets.FloatSlider(

    value=55, min=5, max=85, step=5,

    description="θ (°):", continuous_update=False

)



run_button = widgets.Button(description="Run", button_style="primary", icon="play")

reset_button = widgets.Button(description="Reset")

value_label = widgets.HTML()

sim_output = widgets.Output(layout=widgets.Layout(width="720px"))





def show_preview():

    """The pitched launch arrow and the parabola trace."""

    value_label.value = summary(speed_slider.value, angle_slider.value)

    show_launch(speed_slider.value, angle_slider.value)





def run_clicked(button):

    value_label.value = summary(speed_slider.value, angle_slider.value)

    with sim_output:

        sim_output.clear_output(wait=True)

        display(projectile_animation(speed_slider.value, angle_slider.value))





def reset_clicked(button):

    with sim_output:

        sim_output.clear_output(wait=True)

        show_preview()





def slider_changed(change):

    """Re-draw the preview every time a slider is released."""

    with sim_output:

        sim_output.clear_output(wait=True)

        show_preview()





run_button.on_click(run_clicked)

reset_button.on_click(reset_clicked)

speed_slider.observe(slider_changed, names="value")

angle_slider.observe(slider_changed, names="value")



controls = widgets.VBox([

    widgets.HTML("<h3>Controls</h3>"),

    speed_slider,

    angle_slider,

    widgets.HBox([run_button, reset_button]),

    value_label

], layout=widgets.Layout(width="330px"))



display(widgets.HBox([controls, sim_output],

                     layout=widgets.Layout(align_items="flex-start", gap="18px")))



with sim_output:

    show_preview()
